In [2]:
import pandas as pd
import numpy as np
import random

# Load the saved dataframes into this new notebook's memory
df_original = pd.read_csv("enriched_lightcurves.csv") # Or however you initially loaded the raw data
df_new = pd.read_pickle("astroplan_ready_targets.pkl")

# NOW you can run your validation test function!

In [6]:
import random
import numpy as np

def run_validation_test(df_original, df_new, num_tests=3):
    """
    Randomly samples rows to ensure coordinates and times match exactly 
    between the original dataframe and the Astroplan-ready dataframe.
    """
    print(f"--- Running {num_tests} Random Validation Tests ---\n")
    
    # Get a list of all possible detection time columns
    time_cols = [col for col in df_original.columns if 'Detection_Time_MJD_' in col]
    
    # Pick random row indices
    total_rows = len(df_original)
    random_indices = random.sample(range(total_rows), num_tests)
    
    all_passed = True
    
    for i, idx in enumerate(random_indices, 1):
        print(f"🔍 Test {i} (Row Index: {idx}):")
        row_orig = df_original.iloc[idx]
        row_new = df_new.iloc[idx]
        
        # --- 1. Coordinate Check ---
        raw_ra = row_orig['RA (deg)']
        raw_dec = row_orig['Dec (deg)']
        
        astro_target = row_new['Astroplan_Target']
        astro_ra = astro_target.coord.ra.deg
        astro_dec = astro_target.coord.dec.deg
        
        # Compare using np.isclose to avoid floating-point micro-differences
        ra_match = np.isclose(raw_ra, astro_ra)
        dec_match = np.isclose(raw_dec, astro_dec)
        
        print(f"   [Coords] Raw: ({raw_ra:.4f}, {raw_dec:.4f}) | Astroplan: ({astro_ra:.4f}, {astro_dec:.4f})")
        print(f"            -> Match: {ra_match and dec_match}")
        if not (ra_match and dec_match): all_passed = False
            
        # --- 2. Random Time Check ---
        # Find all time columns for THIS specific row that aren't empty (NaN)
        valid_time_cols = [col for col in time_cols if pd.notna(row_orig[col])]
        
        if valid_time_cols:
            rand_time_col = random.choice(valid_time_cols)
            raw_mjd = row_orig[rand_time_col]
            astro_time = row_new[rand_time_col]
            
            # Check if astro_time is actually a Time object and compare
            if hasattr(astro_time, 'mjd'):
                time_match = np.isclose(raw_mjd, astro_time.mjd)
                print(f"   [{rand_time_col}] Raw MJD: {raw_mjd:.4f} | Astropy MJD: {astro_time.mjd:.4f}")
                print(f"            -> Match: {time_match}")
                if not time_match: all_passed = False
            else:
                print(f"   [{rand_time_col}] FAILED: Object is not an Astropy Time object! It is {type(astro_time)}")
                all_passed = False
        else:
            print("   [Time] No valid detection times for this specific target to test.")
            
        print("-" * 50)
        
    # --- Final Verdict ---
    if all_passed:
        print("✅ ALL TESTS PASSED! The DataFrame is perfectly synchronized and ready for Astroplan.")
    else:
        print("❌ TEST FAILED! There is a mismatch in the data. Check the logs above.")

# Run the test
run_validation_test(df_original, df_new, num_tests=50)

--- Running 50 Random Validation Tests ---

🔍 Test 1 (Row Index: 1):
   [Coords] Raw: (66.7175, -48.8449) | Astroplan: (66.7175, -48.8449)
            -> Match: True
   [Detection_Time_MJD_83] Raw MJD: 59175.5781 | Astropy MJD: 59175.5781
            -> Match: True
--------------------------------------------------
🔍 Test 2 (Row Index: 116):
   [Coords] Raw: (248.3118, 65.6374) | Astroplan: (248.3118, 65.6374)
            -> Match: True
   [Detection_Time_MJD_24] Raw MJD: 59302.2305 | Astropy MJD: 59302.2305
            -> Match: True
--------------------------------------------------
🔍 Test 3 (Row Index: 21):
   [Coords] Raw: (253.5008, 59.8923) | Astroplan: (253.5008, 59.8923)
            -> Match: True
   [Detection_Time_MJD_26] Raw MJD: 59310.7227 | Astropy MJD: 59310.7227
            -> Match: True
--------------------------------------------------
🔍 Test 4 (Row Index: 89):
   [Coords] Raw: (245.9366, 65.4745) | Astroplan: (245.9366, 65.4745)
            -> Match: True
   [Detecti